### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="santander_transaction_value",
    dataset_year="2018",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/santander-value-prediction-challenge",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c santander-value-prediction-challenge -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/santander_transaction_value && mv train.csv local-data-warehouse/santander_transaction_value/
""",
    # References
    academic_reference_bibtex=r"""@misc{McDonald2018SantanderValuePredictionChallenge,
  author = {Mark McDonald and Mercedes Piedra and Sohier Dane and Soraya Jimenez},
  title  = {Santander Value Prediction Challenge},
  year   = {2018},
  howpublished = {\url{https://kaggle.com/competitions/santander-value-prediction-challenge}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="McDonald2018SantanderValuePredictionChallenge",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized.
- We follow the preprocessing by Tschalzev et al. (https://arxiv.org/abs/2407.02112), which follows https://www.kaggle.com/competitions/santander-value-prediction-challenge/discussion/63919
- Since we restrict to the train.csv, we do not exploit the test leak of the competition.
- The expert preprocessing aggregates over feature groups per row to create the new features for the final dataset. Feature groups for aggregation are found by searching for chains of features (?).
- The data contains a very small number of duplicates. We remove them to avoid leakage in our own splits.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="target",
    problem_type="regression",
    objective_metric_name="rmsle",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

og_df = pd.read_csv(dataset_mold.path / "train.csv")
# print("Loaded data shape:", df.shape)

In [3]:
train = og_df.copy()

# Code adapted from https://github.com/atschalz/dc_tabeval/blob/main/datasets.py#L2120 w/o test leak part
import gc
features = [
    'f190486d6', '58e2e02e6', 'eeb9cd3aa', '9fd594eec', '6eef030c1',
    '15ace8c9f', 'fb0f5dbfe', '58e056e12', '20aa07010', '024c577b9',
    'd6bb78916', 'b43a7cfd5', '58232a6fb', '1702b5bf0', '324921c7b',
    '62e59a501', '2ec5b290f', '241f0f867', 'fb49e4212', '66ace2992',
    'f74e8f13d', '5c6487af1', '963a49cdc', '26fc93eb7', '1931ccfdd',
    '703885424', '70feb1494', '491b9ee45', '23310aa6f', 'e176a204a',
    '6619d81fc', '1db387535', 'fc99f9426', '91f701ba2', '0572565c2',
    '190db8488', 'adb64ff71', 'c47340d97', 'c5a231d81', '0ff32eb98'
]
extra_features = []

train = train
train_t = train.drop(['target'], axis = 1, inplace=False)
train_t.set_index('ID', inplace=True)
train_t = train_t.T

#run this iteratively until you have no more links. Then prune
def chain_pairs(ordered_items):
    ordered_chains = []
    links_found = 0
    for i_1, op_chain in enumerate(ordered_items.copy()[:]):
        if op_chain[0] != op_chain[1]:
            end_chain = op_chain[-1]
            for i_2, op in enumerate(ordered_items.copy()[:]):
                if (end_chain == op[0]):
                    links_found += 1
                    op_chain.extend(op[1:])
                    end_chain = op_chain[-1]

            ordered_chains.append(op_chain)
    return links_found, ordered_chains

def prune_chain(ordered_chain):

    ordered_chain = sorted(ordered_chain, key=len, reverse=True)
    new_chain = []
    id_lookup = {}
    for oc in ordered_chain:
        id_already_in_chain = False
        for idd in oc:
            if idd in id_lookup:
                id_already_in_chain = True
            id_lookup[idd] = idd

        if not id_already_in_chain:
            new_chain.append(oc)
    return sorted(new_chain, key=len, reverse=True)

def find_new_ordered_features(ordered_ids, data_t):
    data = data_t.copy()

    f1 = ordered_ids[0][:-1]
    f2 = ordered_ids[0][1:]
    for ef in ordered_ids[1:]:
        f1 += ef[:-1]
        f2 += ef[1:]

    d1 = data[f1].apply(tuple, axis=1).apply(hash).to_frame().rename(columns={0: 'key'})
    d1['ID'] = data.index
    gc.collect()
    d2 = data[f2].apply(tuple, axis=1).apply(hash).to_frame().rename(columns={0: 'key'})
    d2['ID'] = data.index
    gc.collect()
    d3 = d2[~d2.duplicated(['key'], keep=False)]
    d4 = d1[~d1.duplicated(['key'], keep=False)]
    d5 = d4.merge(d3, how='inner', on='key')

    d_feat = d1.merge(d5, how='left', on='key')
    d_feat.fillna(0, inplace=True)

    ordered_features = list(d_feat[['ID_x', 'ID_y']][d_feat.ID_x != 0].apply(list, axis=1))
    del d1,d2,d3,d4,d5,d_feat
    gc.collect()

    links_found = 1
    while links_found > 0:
        links_found, ordered_features = chain_pairs(ordered_features)

    ordered_features = prune_chain(ordered_features)
    #make lookup of all features found so far
    found = {}
    for ef in extra_features:
        found[ef[0]] = ef
    found [features[0]] = features

    new_feature_sets = []
    for of in ordered_features:
        if len(of) >= 40:
            if of[0] not in found:
                new_feature_sets.append(of)

    return new_feature_sets

def add_new_feature_sets(data, data_t):

    # print ('\nData Shape:', data.shape)
    f1 = features[:-1]
    f2 = features[1:]

    for ef in extra_features:
        f1 += ef[:-1]
        f2 += ef[1:]

    d1 = data[f1].apply(tuple, axis=1).apply(hash).to_frame().rename(columns={0: 'key'})
    d1['ID'] = data['ID']
    gc.collect()
    d2 = data[f2].apply(tuple, axis=1).apply(hash).to_frame().rename(columns={0: 'key'})
    d2['ID'] = data['ID']
    gc.collect()
    #print('here')
    d3 = d2[~d2.duplicated(['key'], keep=False)]
    del d2
    d4 = d1[~d1.duplicated(['key'], keep=False)]
    #print('here')
    d5 = d4.merge(d3, how='inner', on='key')
    del d4
    d = d1.merge(d5, how='left', on='key')
    d.fillna(0, inplace=True)
    #print('here')
    ordered_ids = list(d[['ID_x', 'ID_y']][d.ID_x != 0].apply(list, axis=1))
    del d1,d3,d5,d
    gc.collect()

    links_found = 1
    while links_found > 0:
        links_found, ordered_ids = chain_pairs(ordered_ids)
        #print(links_found)

    # print ('OrderedIds:', len(ordered_ids))
    #Make distinct ordered id chains
    ordered_ids = prune_chain(ordered_ids)
    # print ('OrderedIds Pruned:', len(ordered_ids))

    #look for ordered features with new ordered id chains
    new_feature_sets = find_new_ordered_features(ordered_ids, data_t)

    extra_features.extend(new_feature_sets)
    # print('New Feature Count:', len(new_feature_sets))
    # print('Extra Feature Count:', len(extra_features))

add_new_feature_sets(train,train_t)
add_new_feature_sets(train,train_t)
add_new_feature_sets(train,train_t)

del train_t
gc.collect()

### Create 40 features
extra_features_list = []

for ef in extra_features:
    extra_features_list.extend(ef)

extra_features_list.extend(features)

#This makes the 100 40 length feature groups into 40 100 length feature groups.
feats = pd.DataFrame(extra_features)
time_features = []
for c in feats.columns[:]:
    time_features.append([f for f in feats[c].values if f is not None])


#Make a bunch of different feature groups to build aggregates from
agg_features = []
all_cols = train.columns.drop(['ID', 'target'])
agg_features.append(all_cols)
agg_features.append([c for c in all_cols if c not in extra_features_list])
agg_features.append(extra_features_list)
agg_features.extend(time_features)
agg_features.extend(extra_features)

def add_new_features(source, dest, feats):
    high = source[feats].max(axis=1)
    high.name = 'high_{}_{}'.format(feats[0], len(feats))
    mean = source[feats].replace(0, np.nan).mean(axis=1)
    mean.name = 'mean_{}_{}'.format(feats[0], len(feats))
    low = source[feats].replace(0, np.nan).min(axis=1)
    low.name = 'low_{}_{}'.format(feats[0], len(feats))
    median = source[feats].replace(0, np.nan).median(axis=1)
    median.name = 'median_{}_{}'.format(feats[0], len(feats))
    sum = source[feats].sum(axis=1)
    sum.name = 'sum_{}_{}'.format(feats[0], len(feats))
    stddev = source[feats].std(axis=1)
    stddev.name = 'stddev_{}_{}'.format(feats[0], len(feats))
    first_nonZero= np.log1p(source[feats].replace(0, np.nan).bfill(axis=1).iloc[:, 0])
    first_nonZero.name = 'first_nonZero_{}_{}'.format(feats[0], len(feats))
    last_nonZero = np.log1p(source[feats[::-1]].replace(0, np.nan).bfill(axis=1).iloc[:, 0])
    last_nonZero.name = 'last_nonZero_{}_{}'.format(feats[0], len(feats))
    nb_nans =  source[feats].replace(0, np.nan).isnull().sum(axis=1)
    nb_nans.name = 'nb_nans_{}_{}'.format(feats[0], len(feats))
    unique = source[feats].nunique(axis=1)
    unique.name = 'unique_{}_{}'.format(feats[0], len(feats))

    dest = pd.concat([dest, high, mean, low, median, sum, stddev, first_nonZero, last_nonZero, nb_nans, unique],axis=1)
    return dest


train_feats = pd.DataFrame()
for i, ef in list(enumerate(agg_features)):
    train_feats = add_new_features(train, train_feats, ef)


df = train_feats
df[task_mold.target_column_name] = np.log1p(train.target.values)

# Remove duplicates
df = df.drop_duplicates(subset=[c for c in df.columns if c != task_mold.target_column_name])
df = df.reset_index(drop=True)

## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 4,447
Columns: 541
Use sampling: False (sample size: 4,447)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['stddev_48df886f9_4511', 'stddev_48df886f9_4991', 'mean_48df886f9_4991', 'sum_48df886f9_4991', 'mean_48df886f9_4511', 'sum_48df886f9_4511', 'stddev_9ddd6d137_480', 'mean_9ddd6d137_480', 'sum_9ddd6d137_480', 'median_48df886f9_4991']
Rows remaining as candidates after top-10 filter: 94 (of 4,447)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,high_48df886f9_4991,mean_48df886f9_4991,low_48df886f9_4991,median_48df886f9_4991,sum_48df886f9_4991,stddev_48df886f9_4991,first_nonZero_48df886f9_4991,last_nonZero_48df886f9_4991,nb_nans_48df886f9_4991,unique_48df886f9_4991,high_48df886f9_4511,mean_48df886f9_4511,low_48df886f9_4511,median_48df886f9_4511,sum_48df886f9_4511,stddev_48df886f9_4511,first_nonZero_48df886f9_4511,last_nonZero_48df886f9_4511,nb_nans_48df886f9_4511,unique_48df886f9_4511,high_9ddd6d137_480,mean_9ddd6d137_480,low_9ddd6d137_480,median_9ddd6d137_480,sum_9ddd6d137_480,stddev_9ddd6d137_480,first_nonZero_9ddd6d137_480,last_nonZero_9ddd6d137_480,nb_nans_9ddd6d137_480,unique_9ddd6d137_480,high_9ddd6d137_11,mean_9ddd6d137_11,low_9ddd6d137_11,median_9ddd6d137_11,sum_9ddd6d137_11,stddev_9ddd6d137_11,first_nonZero_9ddd6d137_11,last_nonZero_9ddd6d137_11,nb_nans_9ddd6d137_11,unique_9ddd6d137_11,high_5cfc625f1_11,mean_5cfc625f1_11,low_5cfc625f1_11,median_5cfc625f1_11,sum_5cfc625f1_11,stddev_5cfc625f1_11,first_nonZero_5cfc625f1_11,last_nonZero_5cfc625f1_11,nb_nans_5cfc625f1_11,unique_5cfc625f1_11,high_8984e4066_11,mean_8984e4066_11,low_8984e4066_11,median_8984e4066_11,sum_8984e4066_11,stddev_8984e4066_11,first_nonZero_8984e4066_11,last_nonZero_8984e4066_11,nb_nans_8984e4066_11,unique_8984e4066_11,high_0ccd6454a_11,mean_0ccd6454a_11,low_0ccd6454a_11,median_0ccd6454a_11,sum_0ccd6454a_11,stddev_0ccd6454a_11,first_nonZero_0ccd6454a_11,last_nonZero_0ccd6454a_11,nb_nans_0ccd6454a_11,unique_0ccd6454a_11,high_9397535c7_11,mean_9397535c7_11,low_9397535c7_11,median_9397535c7_11,sum_9397535c7_11,stddev_9397535c7_11,first_nonZero_9397535c7_11,last_nonZero_9397535c7_11,nb_nans_9397535c7_11,unique_9397535c7_11,high_de7063efa_11,mean_de7063efa_11,low_de7063efa_11,median_de7063efa_11,sum_de7063efa_11,stddev_de7063efa_11,first_nonZero_de7063efa_11,last_nonZero_de7063efa_11,nb_nans_de7063efa_11,unique_de7063efa_11,high_74f3ac6af_11,mean_74f3ac6af_11,low_74f3ac6af_11,median_74f3ac6af_11,sum_74f3ac6af_11,stddev_74f3ac6af_11,first_nonZero_74f3ac6af_11,last_nonZero_74f3ac6af_11,nb_nans_74f3ac6af_11,unique_74f3ac6af_11,high_6bee3733e_11,mean_6bee3733e_11,low_6bee3733e_11,median_6bee3733e_11,sum_6bee3733e_11,stddev_6bee3733e_11,first_nonZero_6bee3733e_11,last_nonZero_6bee3733e_11,nb_nans_6bee3733e_11,unique_6bee3733e_11,high_20e2c484e_11,mean_20e2c484e_11,low_20e2c484e_11,median_20e2c484e_11,sum_20e2c484e_11,stddev_20e2c484e_11,first_nonZero_20e2c484e_11,last_nonZero_20e2c484e_11,nb_nans_20e2c484e_11,unique_20e2c484e_11,high_5adfe7419_11,mean_5adfe7419_11,low_5adfe7419_11,median_5adfe7419_11,sum_5adfe7419_11,stddev_5adfe7419_11,first_nonZero_5adfe7419_11,last_nonZero_5adfe7419_11,nb_nans_5adfe7419_11,unique_5adfe7419_11,high_03a4ccd7c_11,mean_03a4ccd7c_11,low_03a4ccd7c_11,median_03a4ccd7c_11,sum_03a4ccd7c_11,stddev_03a4ccd7c_11,first_nonZero_03a4ccd7c_11,last_nonZero_03a4ccd7c_11,nb_nans_03a4ccd7c_11,unique_03a4ccd7c_11,high_ecbd077d0_11,mean_ecbd077d0_11,low_ecbd077d0_11,median_ecbd077d0_11,sum_ecbd077d0_11,stddev_ecbd077d0_11,first_nonZero_ecbd077d0_11,last_nonZero_ecbd077d0_11,nb_nans_ecbd077d0_11,unique_ecbd077d0_11,high_851697562_11,mean_851697562_11,low_851697562_11,median_851697562_11,sum_851697562_11,stddev_851697562_11,first_nonZero_851697562_11,last_nonZero_851697562_11,nb_nans_851697562_11,unique_851697562_11,high_60cb16e88_11,mean_60cb16e88_11,low_60cb16e88_11,median_60cb16e88_11,sum_60cb16e88_11,stddev_60cb16e88_11,first_nonZero_60cb16e88_11,last_nonZero_60cb16e88_11,nb_nans_60cb16e88_11,unique_60cb16e88_11,high_73a8a4d75_11,mean_73a8a4d75_11,low_73a8a4d75_11,median_73a8a4d75_11,sum_73a8a4d75_11,stddev_73a8a4d75_11,first_nonZero_73a8a4d75_11,last_nonZero_73a8a4d75_11,nb_nans_73a8a4d75_11,unique_73a8a4d75_11,high_4c48708d8_11,mean_4c48708d8_11,low_4c48708d8_11,median_4c48708d8_11,sum_4c48708d8_11,stddev_4c48708d8_11,first_nonZero_4c48708d8_11,last_nonZero_4c48708d8_11,nb_nans_4c48708d8_11,unique_4c48708d8_11,high_ea72c62a1_11,mean_ea72c62a1_11,low_ea72c62a1_11,median_ea72c62a1_11,

In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,mean_d48c44c49_40,float64,4174.0,93.86,67.0,"2000000.0, 5618000.0, 260000.0, 1600000.0, 200000.0, 9000000.0, 1280000.0, 320000.0, 70000.0, 310000.0"
1,low_d48c44c49_40,float64,4174.0,93.86,29.0,"252000.0, 2000000.0, 5618000.0, 320000.0, 160000.0, 300000.0, 1600000.0, 200000.0, 9000000.0, 1280000.0"
2,median_d48c44c49_40,float64,4174.0,93.86,34.0,"1032000.0, 2000000.0, 5618000.0, 240000.0, 1600000.0, 300000.0, 200000.0, 1280000.0, 9000000.0, 320000.0"
3,first_nonZero_d48c44c49_40,float64,4174.0,93.86,32.0,"12.6761, 14.5087, 13.8235, 15.5415, 11.9829, 12.6115, 14.2855, 12.2061, 16.0127, 14.0624"
4,last_nonZero_d48c44c49_40,float64,4174.0,93.86,29.0,"13.847, 12.6115, 14.5087, 15.5415, 12.6761, 14.2855, 12.2061, 16.0127, 14.0624, 13.3047"
5,mean_9ddd6d137_40,float64,4098.0,92.15,57.0,"1360000.0, 20106000.0, 33333333.3333, 220000.0, 120000000.0, 1244000.0, 10000000.0, 5200000.0, 236000.0, 46000000.0"
6,low_9ddd6d137_40,float64,4098.0,92.15,30.0,"220000.0, 1360000.0, 20106000.0, 456000.0, 20000000.0, 252000.0, 120000000.0, 8000.0, 10000000.0, 5200000.0"
7,median_9ddd6d137_40,float64,4098.0,92.15,39.0,"1360000.0, 20106000.0, 30000000.0, 472000.0, 220000.0, 120000000.0, 22000.0, 10000000.0, 5200000.0, 236000.0"
8,first_nonZero_9ddd6d137_40,float64,4098.0,92.15,33.0,"13.0303, 14.123, 16.8165, 16.8112, 12.3014, 18.603, 8.9873, 12.4372, 16.1181, 15.4642"
9,last_nonZero_9ddd6d137_40,float64,4098.0,92.15,29.0,"12.3014, 14.123, 16.8165, 17.7275, 12.4372, 18.603, 9.9988, 13.0647, 16.1181, 15.4642"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
high_48df886f9_4991,4447.0,9.093418e+07,2.481567e+08,4000.000000,4.652000e+09
mean_48df886f9_4991,4447.0,9.846985e+06,1.460954e+07,4000.000000,4.000000e+08
low_48df886f9_4991,4447.0,1.249765e+06,8.502263e+06,52.000000,4.000000e+08
median_48df886f9_4991,4447.0,6.077197e+06,1.344192e+07,2000.000000,4.000000e+08
sum_48df886f9_4991,4447.0,1.392292e+09,2.408176e+09,4000.000000,2.273642e+10
stddev_48df886f9_4991,4447.0,2.783841e+06,4.538350e+06,56.619523,6.594061e+07
first_nonZero_48df886f9_4991,4447.0,1.441295e+01,2.161032e+00,3.970292,2.021244e+01
last_nonZero_48df886f9_4991,4447.0,1.452458e+01,2.123884e+00,6.511745,2.021244e+01
nb_nans_48df886f9_4991,4447.0,4.833572e+03,2.122190e+02,3002.000000,4.990000e+03
unique_48df886f9_4991,4447.0,6.715426e+01,1.066218e+02,2.000000,8.130000e+02


In [8]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [9]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.424,-0.665,3.062,0.016,log,43548.1,3.659291e+16,exponential


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to santander_transaction_value/019d5dca-1ba7-77c8-b93c-71d396f67068
019d5dca-1ba7-77c8-b93c-71d396f67068
4604a2f3a2f182583102cbe78711d66e7b4296e07da8b97e0dd751b7a1cff504
